# Submission 06 - XGBoost + LightGBM Blend

Winning Experiment 16 blend:

- XGBoost: 70%
- LightGBM: 30%

Experiment 16 validation ROC-AUC: **0.941815**

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path(r'C:\Users\aakif\Documents\DataCompetition')
TRAIN_PATH = PROJECT_ROOT / 'data' / 'train.csv'
TEST_PATH = PROJECT_ROOT / 'data' / 'test.csv'
SAMPLE_PATH = PROJECT_ROOT / 'data' / 'sample_submission.csv'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'
SUBMISSION_PATH = SUBMISSION_DIR / 'submission_06.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

X = train.drop(columns=['Will_Buy_EV', 'id'])
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})
X_test = test.drop(columns=['id'])

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(X_test)

print('Training shape:', X_processed.shape)
print('Test shape:', X_test_processed.shape)

Training shape: (668665, 24)
Test shape: (286571, 24)


In [2]:
xgb_model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

lgb_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.04,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=0.0,
    objective='binary',
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

print('Training XGBoost...')
xgb_model.fit(X_processed, y)

print('Training LightGBM...')
lgb_model.fit(X_processed, y)

print('Both models trained on the full training dataset.')

Training XGBoost...
Training LightGBM...
Both models trained on the full training dataset.


In [3]:
xgb_pred = xgb_model.predict_proba(X_test_processed)[:, 1]
lgb_pred = lgb_model.predict_proba(X_test_processed)[:, 1]

final_pred = (0.70 * xgb_pred) + (0.30 * lgb_pred)

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

if SAMPLE_PATH.exists():
    sample_submission = pd.read_csv(SAMPLE_PATH)
    submission = sample_submission.copy()
    target_columns = [column for column in submission.columns if column != 'id']

    if len(target_columns) != 1:
        raise ValueError(
            f'Expected exactly one target column in sample_submission.csv, found: {target_columns}'
        )

    target_column = target_columns[0]
    submission[target_column] = final_pred
else:
    submission = pd.DataFrame({
        'id': test['id'],
        'Will_Buy_EV': final_pred
    })

submission.to_csv(SUBMISSION_PATH, index=False)

print('=' * 60)
print('SUBMISSION 06 CREATED')
print('=' * 60)
print(f'File: {SUBMISSION_PATH}')
print(f'Rows: {len(submission)}')
print(f'Columns: {submission.columns.tolist()}')
print()
print(submission.head())

SUBMISSION 06 CREATED
File: C:\Users\aakif\Documents\DataCompetition\submissions\submission_06.csv
Rows: 286571
Columns: ['id', 'Will_Buy_EV']

       id  Will_Buy_EV
0  668665     0.009760
1  668666     0.016574
2  668667     0.005383
3  668668     0.003239
4  668669     0.020122


In [4]:
check = pd.read_csv(SUBMISSION_PATH)

assert len(check) == len(test), 'Submission row count does not match test.csv.'
assert check['id'].equals(test['id']), 'Submission IDs do not match test.csv.'
assert check.iloc[:, 1].notna().all(), 'Submission contains missing predictions.'
assert ((check.iloc[:, 1] >= 0) & (check.iloc[:, 1] <= 1)).all(), 'Predictions must be probabilities between 0 and 1.'

print('Submission validation passed.')
print(f'Prediction minimum: {check.iloc[:, 1].min():.6f}')
print(f'Prediction maximum: {check.iloc[:, 1].max():.6f}')

Submission validation passed.
Prediction minimum: 0.000007
Prediction maximum: 0.969741
